In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [16]:
import os

import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('ignore')

# module_path = os.path.abspath(os.path.join('..'))
# if module_path not in sys.path:
#     sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics         import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics         import roc_auc_score
from sklearn.metrics         import precision_recall_curve, classification_report
from sklearn.datasets         import make_classification

# Model import
from sklearn.tree           import DecisionTreeClassifier
from sklearn.ensemble       import RandomForestClassifier
from sklearn.ensemble       import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.linear_model   import LinearRegression
from sklearn.svm            import SVC
from sklearn.metrics        import classification_report
from xgboost                import XGBClassifier
from xgboost                import plot_importance
from lightgbm               import LGBMClassifier
from catboost               import CatBoostClassifier

# hyperopt 용
from hyperopt               import hp

# 사용자 Functions import
import HyperParams          as HP 
import utils.data_sampling  as ds 

from utils import user_utils    as uu
from utils import preprocessing as pp
from utils import data_sampling as ds
from utils import model_utils   as mu
from utils import modeling      as mo

In [3]:
# 결과받을 딕셔너리
results = {}

In [4]:
#1. 데이터 로딩
raw_df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [5]:
# 2. Time 컬럼 삭제 , 데이터,타겟 분리
X_features, y_target = pp.split_features_target(raw_df, cols= 'Time')

In [6]:
# 3. 이상치를 경계값으로 치환
cap_X_feature = pp.cap_outliers(X_features)

In [7]:
# 4.1 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = pp.data_split(cap_X_feature, y_target)

In [8]:
# 4.2 Over Sampling 하는 경우
X_over, y_over = ds.oversampling_smote(X_train, y_train)

✅ SMOTE 오버샘플링 완료
   원본 샘플 수: 227845 (Class 0: 227451, Class 1: 394)
   샘플링 후: 454902 (Class 0: 227451, Class 1: 227451)


In [9]:
# 5.1 학습/검증 데이터 분리
X_tr, X_val, y_tr, y_val = pp.data_split(X_train, y_train, size=0.4)

In [10]:
# 5.2 Over Sampling한 경우 학습/검증 데이터 분리
X_tr, X_val, y_tr, y_val = pp.data_split(X_over, y_over, size=0.4)

In [11]:
#6 하이퍼파라미터
tuner = uu.HyperOptTuner(max_evals=100, random_state=23)

In [ ]:
# 모델별 스페이스 생성 : 예시는 catboost 
# learning_rate (0.01–0.2), max_depth (3–10), n_estimators (100–1000), subsample (0.5–1.0), colsample_bytree (0.5–1.0)    
xgb_search_space = {
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),
    'subsample': hp.quniform('subsample', 0.5, 1.0, 0.1),  
    'max_depth': hp.quniform('max_depth', 3, 10, 1), 
    'learning_rate': hp.loguniform('learning_rate', np.log(0.001), np.log(0.3)),  # 수정: loguniform이 더 적합
    'colsample_bytree': hp.quniform('colsample_bytree', 0.5, 1.0, 0.1),  
    'scale_pos_weight': hp.quniform('scale_pos_weight', 1, 100, 1),  
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),  
    'gamma': hp.uniform('gamma', 0, 5),
    'reg_alpha': hp.loguniform('reg_alpha', np.log(0.001), np.log(10)),  # 수정: 0 제외, 범위 확대
    'reg_lambda': hp.loguniform('reg_lambda', np.log(0.001), np.log(10)),  # 수정: loguniform 사용
}

# 모델 생성
xgb_clf = XGBClassifier()

# 파라미터 지정
best_params, best_xgm, trials, exec_time = tuner.tune(
    xgb_clf, X_tr, y_tr, X_val, y_val, xgb_search_space
)

# best모델로 결과출력
# 모델명 규칙 : 2~3자리 모델명 + _ho_best
model_name = 'xgb_ho_best2'
results[model_name] =uu.get_model_train_eval(
    best_xgm, model_name, X_train, X_test, y_train, y_test, best_params
)



XGBClassifier 튜닝 시작
100%|██████████| 100/100 [16:19<00:00,  9.79s/trial, best loss: -1.0]             

튜닝 시간: 979.35초
최적 recall: 1.0000

최적 모델의 전체 평가 점수:
- roc_auc: 0.9995
- f1: 0.9758
- precision: 0.9528
- recall: 1.0000
- accuracy: 0.9752

최적 하이퍼파라미터:
-colsample_bytree: 0.6000000000000001
-gamma: 4.787185438562684
-learning_rate: 0.005262189759984318
-max_depth: 5
-min_child_weight: 6
-n_estimators: 700
-reg_alpha: 0.020391918886446342
-reg_lambda: 0.0969098930319443
-scale_pos_weight: 14.0
-subsample: 0.9
-random_state: 23
✓ 모델 저장 완료: ../models\xgb_ho_best2.pkl
  파일 크기: 1.47 MB
folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'AUC': 0.9731, '정확도': 0.9994, '정밀도': 0.8100, '재현율': 0.8265, 'F1': 0.8182 }
{'오차행렬':
[[56845    19]
 [   17    81]] }
실행 시간: 6.364752531051636
하이퍼파라미터: {'colsample_bytree': 0.6000000000000001, 'gamma': 4.787185438562684, 'learning_rate': 0.005262189759984318, 'max_depth': 5, 'min_child_weight': 6, 'n_estimators': 700, 'reg_alpha': 0.02039

In [17]:
# SVC
from sklearn.svm            import LinearSVC

lsvc_model = LinearSVC(random_state=23)

lsvc_search_space = {
    'C': hp.loguniform('C', np.log(0.001), np.log(1000)),  # 정규화 강도 (작을수록 강한 정규화)
    'class_weight': hp.choice('class_weight', [None, 'balanced']),  # 클래스 가중치
    'max_iter': hp.quniform('max_iter', 1000, 10000, 1000),  # 최대 반복 횟수
    'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-2)),  # 수렴 허용 오차
    'dual': hp.choice('dual', [False, True]),  # dual formulation (n_samples > n_features일 때 False 권장)
}

# 파라미터 지정
best_params, best_catboost, trials, exec_time = tuner.tune(
    lsvc_model, X_tr, y_tr, X_val, y_val, lsvc_search_space
)


LinearSVC 튜닝 시작
Error in objective function: The 'max_iter' parameter of LinearSVC must be an int in the range [0, inf). Got 1000.0 instead.
Error in objective function: The 'max_iter' parameter of LinearSVC must be an int in the range [0, inf). Got 2000.0 instead.
Error in objective function: The 'max_iter' parameter of LinearSVC must be an int in the range [0, inf). Got 2000.0 instead.
Error in objective function: The 'max_iter' parameter of LinearSVC must be an int in the range [0, inf). Got 3000.0 instead.
Error in objective function: The 'max_iter' parameter of LinearSVC must be an int in the range [0, inf). Got 10000.0 instead.
Error in objective function: The 'max_iter' parameter of LinearSVC must be an int in the range [0, inf). Got 9000.0 instead.
Error in objective function: The 'max_iter' parameter of LinearSVC must be an int in the range [0, inf). Got 2000.0 instead.
Error in objective function: The 'max_iter' parameter of LinearSVC must be an int in the range [0, inf). Go

KeyError: 'scores'

In [ ]:
# 7 시각화
mo.model_metrics_graph(results, 'cb모델 성능지표 비교')

In [ ]:
# BestOpt 찾고 나서 스케일적용 버전 만들어서 모델링하기 
# X_train_scaled, X_test_scaled, scaler = pp.scale_data(X_train, X_test)